In [ ]:
%matplotlib inline
%reload_ext autoreload
%autoreload 2

work_folder = '/g/stegle/spiljak/benchmarking-cellina/results'
cellina_reproducibility_folder ='/g/stegle/spiljak/cellina_tutorial/cellina-reproducibility/'

import sys
from cellina import CellinaGCN as CellinaModel
from cellina._spatial_utils import spatial_neighbors
from scib_metrics.benchmark import Benchmarker, BioConservation, BatchCorrection
from scvi.train._callbacks import SaveCheckpoint, EarlyStopping

sys.path.append(f'{cellina_reproducibility_folder}/notebooks/application')
from helpers import _normalize_counts, safe_log2_fold_change, compute_correlations, subsample_adata

sys.path.append(f'{cellina_reproducibility_folder}/scripts')
import numpy as np
import scanpy as sc
import anndata as ad
import pandas as pd
import os
import seaborn as sns
import matplotlib.pyplot as plt

from tqdm import tqdm
from sklearn.model_selection import train_test_split

#from cellina_graph import CellinaModel
from utils import set_seed
from train_loo import preprocess_adata
from scripts.perturb_utils import load_crc_slide
  

slide_ids = [110, 120, 210, 221, 222, 231, 232, 242]


#get dataset -----------------------------------
for slide_id in slide_ids:

    set_seed(0)
    adata = load_crc_slide(slide_id=slide_id, data_dir=f"{work_folder}/samples")  
    #adata = subsample_adata(adata, fraction=0.05)
    
    slide_id = str(slide_id)
    spatial_neighbors(adata, bandwidth=100, max_neighbours=20, standardize=False, ) #adds spatial_connectivites   ? this version doesn't support um_neighbours=[-1]

    labels_key = 'coarse_type'
    domains_key = 'typ_clean'
    batch_key = 'sid'

    #data splitting -----------------------------------

    split = "random"

    # Get holdout indices
    if split == "random":
        fraction = 0.1
        n_cells = adata.n_obs
        n_holdout = int(n_cells * fraction)

        # Randomly choose cells
        test_idx = np.random.choice(n_cells, n_holdout, replace=False)

    elif split == "ood":
        holdout_ct = "Fibroblast"
        is_tumor_region  = adata.obs[domains_key].str.contains("CRC", regex=True)
        is_holdout_ct = adata.obs[labels_key] == holdout_ct

        # Combine for test set
        test_mask = (is_tumor_region) & (is_holdout_ct)
        test_idx = np.where(test_mask)[0]
    else:
        raise ValueError(f"Unknown split: {split}")

    # Get train/val indices
    all_idx = np.arange(adata.n_obs)
    trainval_idx = np.setdiff1d(all_idx, test_idx)

    # Set 'is_holdout' to False by default, then True for selected cells
    adata.obs['is_holdout'] = False
    adata.obs.iloc[test_idx, adata.obs.columns.get_loc('is_holdout')] = True

    validation_size = 0.1
    train_idx, val_idx = train_test_split(
        trainval_idx,
        test_size=validation_size,
        random_state=0,
        shuffle=True,
    )

    # training -----------------------------------

    model_base_path = f"{work_folder}/{slide_id}"


    model_args = {
        'adata': adata,
        'n_latent': 64,
        'n_layers': 3,
        'use_observed_lib_size': True,
        'condition_on_intrinsic': False,
        'gene_likelihood': 'nb',
        'classifier_lambda': 1.,
        'discriminator_lambda': 1.,
        'link_prediction_weight': 0.1, 
        'num_neighbors': [-1],
        }
    train_args = {'max_epochs': 20,
                'batch_size': 2048,
                'check_val_every_n_epoch': 1,
                'early_stopping': True,
                'devices': [0],
                'datasplitter_kwargs': {
                    "external_indexing": [train_idx, val_idx, test_idx],
                    },
                'enable_checkpointing':True,
                'callbacks': [
                    SaveCheckpoint(
                        monitor='vae_loss_validation',
                        dirpath=f"{model_base_path}",
                        load_best_on_end=True,
                        ),
                    EarlyStopping(
                        monitor="vae_loss_validation",
                        patience=5,
                        mode="min",
                        ),
                    ],
        }

    plan_kwargs = {
        'lr': 1e-3,
        'normalize_losses': True
        }

    CellinaModel.setup_anndata(adata,
                            batch_key=batch_key,
                            labels_key=labels_key, 
                            domains_key=domains_key, 
                            spatial_connectivities_key="spatial_connectivities",
                            layer='counts')

    model = CellinaModel(**model_args, )
    model.train(**train_args, plan_kwargs=plan_kwargs)


    # inference and saving adata with latent space -----------------------------------


    checkpoint_name = next(
        d for d in os.listdir(model_base_path)
        if d.startswith("epoch=")
    )
    model = CellinaModel.load(
        f'{work_folder}/{slide_id}/{checkpoint_name}',
        adata=adata,
    )

    adata.obsm['cellina_basal'] = model.get_latent_representation(adata=adata, latent_key='z', batch_size=2048)
    adata.obsm['cellina_spatial'] = model.get_latent_representation(adata=adata, latent_key='s', batch_size=2048)


    os.makedirs(work_folder, exist_ok=True)   # creates it; silently does nothing if it exists
    output_dir = os.path.join(work_folder, slide_id)
    os.makedirs(output_dir, exist_ok=True)   # creates it; silently does nothing if it exists
    adata.write_h5ad(f"{output_dir}/latents_adata.h5ad") 
    model.save(f"{output_dir}", overwrite=True)

  0%|          | 0.00/1.41G [00:00<?, ?B/s]

/g/stegle/spiljak/benchmarking-cellina/results/samples/crc_110.h5ad


/g/stegle/spiljak/programs/miniforge3/envs/cellina_edge/lib/python3.10/site-packages/scvi/data/fields/_layer_field.py:115: UserWarning: Training will be faster when sparse matrix is formatted as CSR. It is safe to cast before model initialization.
  _verify_and_correct_data_format(adata, self.attr_name, self.attr_key)
/g/stegle/spiljak/cellina/src/cellina/_cellina_gcn_model.py:724: UserWarning: len(num_neighbors)=1 != n_layers=3; NeighborLoader samples len(num_neighbors) hops differing from the GCN depth.Pass a length-3 list to silence. Got [-1].
  warnings.warn(


INFO     cellina: The CellinaGCN model has been initialized with adversarial domain forgetting with edge prediction


/g/stegle/spiljak/programs/miniforge3/envs/cellina_edge/lib/python3.10/site-packages/lightning/fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /g/stegle/spiljak/programs/miniforge3/envs/cellina_e ...
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/g/stegle/spiljak/programs/miniforge3/envs/cellina_edge/lib/python3.10/site-packages/lightning/fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /g/stegle/spiljak/programs/miniforge3/envs/cellina_e ...
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [3]
/g/stegle/spiljak/programs/miniforge3/envs/cellina_edge/lib/python3.10/site-packages/torch_geome

Training:   0%|          | 0/20 [00:00<?, ?it/s]